# Cell 1 — Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import pickle
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Cell 2 — Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Cell 3 — Load Dataset

In [3]:
with open("../data/processed/dl/dl_train_test_split.pkl", "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]

y_train = data["y_train"]
y_test = data["y_test"]

print(X_train.shape)

(1539641, 103)


# Cell 4 — Feature Scaling

The Edge-IIoTset preprocessing pipeline applies standardization before training models.

In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Cell 5 — Label Encoding

In [5]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)
print("Classes:", num_classes)

Classes: 15


# Cell 6 — Reshape Data for CNN

CNN expects:

`(batch, channels, sequence_length)`

Example:

`(1700000, 1, 59)`

In [6]:
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

print(X_train.shape)

(1539641, 1, 103)


# Cell 7 — Convert to Tensor

In [7]:
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)

# Cell 8 — DataLoader

In [8]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

# Cell 9 — CNN Model

Simple 1D CNN architecture.

In [9]:
class CNNModel(nn.Module):

    def __init__(self, input_length, num_classes):

        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv1d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)

        self.pool = nn.MaxPool1d(2)

        self.relu = nn.ReLU()

        conv_output = ((input_length - 2) // 2 - 2) // 2

        self.fc1 = nn.Linear(conv_output * 64, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):

        x = self.relu(self.conv1(x))
        x = self.pool(x)

        x = self.relu(self.conv2(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# Cell 10 — Initialize Model

In [10]:
input_length = X_train.shape[2]

model = CNNModel(input_length, num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

# Cell 11 — Training Loop

In [11]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss}")

Epoch 1/10 Loss: 746.8832368534058
Epoch 2/10 Loss: 623.1155425012112
Epoch 3/10 Loss: 606.4454554561526
Epoch 4/10 Loss: 598.9268937241286
Epoch 5/10 Loss: 597.4540635291487
Epoch 6/10 Loss: 593.9249651115388
Epoch 7/10 Loss: 591.1255966387689
Epoch 8/10 Loss: 589.8365087993443
Epoch 9/10 Loss: 590.5574182569981
Epoch 10/10 Loss: 588.916372269392


# Cell 12 — Evaluation

In [14]:
model.eval()

# Use a DataLoader to process the test set in batches
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

all_preds = []
all_true = []

print("Running GPU evaluation in batches...")
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(y_batch.cpu().numpy())

import numpy as np
y_pred = np.array(all_preds)
y_true = np.array(all_true)

print("✅ Evaluation complete.")


Running GPU evaluation in batches...
✅ Evaluation complete.


# Cell 13 — Accuracy

Expected range from literature:

95% – 97%

CNN-based IDS models commonly reach ~94–97% on Edge-IIoTset-like datasets.

In [15]:
accuracy = accuracy_score(y_true, y_pred)

print("CNN Accuracy:", accuracy)

CNN Accuracy: 0.9515342507748544


# Cell 14 — Classification Report

In [16]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.95      0.96      4806
           1       0.93      0.67      0.78      9871
           2       1.00      1.00      1.00     13588
           3       0.99      1.00      0.99     10013
           4       1.00      1.00      1.00     24314
           5       0.68      0.52      0.59       171
           6       1.00      1.00      1.00        72
           7       1.00      1.00      1.00    275611
           8       0.43      0.82      0.57      9987
           9       0.97      0.99      0.98      3996
          10       0.99      0.96      0.98      1938
          11       0.61      0.27      0.37     10165
          12       0.58      0.47      0.52      7361
          13       1.00      0.84      0.91     10005
          14       0.50      0.93      0.65      3013

    accuracy                           0.95    384911
   macro avg       0.84      0.83      0.82    384911
weighted avg       0.96   

# Cell 15 — Save Model

In [17]:
torch.save(
    model.state_dict(),
    "../models/dl/cnn_model.pth"
)